In [1]:
import pandas as pd
import pandera as pa
import random
from datetime import datetime, timedelta

# Parameter
n_rows = 200  # jumlah baris data
start_date = datetime(2023, 5, 1)
categories = ["Elektronik", "Electronics", "elektronik", "Fashion", "Fesyen"]
statuses = ["paid", "pending", "cancelled"]
emails = ["@mail.com", "@gmail.com", "@yahoo.com", ""]

data = []
for i in range(5001, 5001 + n_rows):
    cust_id = f"C{str(random.randint(1, 50)).zfill(3)}"
    prod_id = f"P{str(random.randint(1, 30)).zfill(3)}" if random.random() > 0.05 else ""  # 5% missing
    
    # tanggal (ada invalid & salah format)
    if random.random() < 0.05:
        tanggal = "2023-02-31"  
    elif random.random() < 0.05:
        tanggal = (start_date + timedelta(days=random.randint(0, 30))).strftime("%d/%m/%Y")
    else:
        tanggal = (start_date + timedelta(days=random.randint(0, 30))).strftime("%Y-%m-%d")
    
    jumlah = random.choice([random.randint(1, 5), -1, 0]) if random.random() < 0.1 else random.randint(1, 5)
    harga = random.choice([0, 99999999, random.randint(100, 1_000_000)]) if random.random() < 0.1 else random.randint(100, 1_000_000)
    
    if random.random() < 0.1:
        email = f"user{i}mail.com"  
    elif random.random() < 0.05:
        email = ""  
    else:
        email = f"user{i}{random.choice(emails)}"
    
    kode_pos = str(random.randint(10000, 99999)) if random.random() > 0.1 else random.choice(["4012a", ""])
    status = random.choice(statuses)
    kategori = random.choice(categories)
    
    data.append([i, cust_id, prod_id, tanggal, jumlah, harga, email, kode_pos, status, kategori])

# Buat DataFrame
df = pd.DataFrame(data, columns=["order_id","customer_id","product_id","tanggal","jumlah","harga","email","kode_pos","status","kategori"])

# Tambahkan duplikat
duplicates = df.sample(5)
df = pd.concat([df, duplicates], ignore_index=True)

# Simpan CSV
df.to_csv("orders_large.csv", index=False)
print("✅ Dataset berhasil dibuat: orders_large.csv")



✅ Dataset berhasil dibuat: orders_large.csv


Setelah melakukan generate dataset, lakukan langkah-langkah berikut:

Question 1. Data Profiling

Instruksi:

    Hitung jumlah record total.
    Hitung jumlah missing value (per kolom).
    Hitung jumlah duplicate order_id.
    Deteksi nilai invalid yang meliputi:
        jumlah ≤ 0
        harga ≤ 0 atau > 10.000.000
        tanggal salah format atau tidak valid
        email tanpa @
    Buat distribusi sederhana untuk jumlah dan harga (bisa histogram, boxplot, atau statistik deskriptif).

Output yang diharapkan:

    Buatlah tabel ringkasan profil data otomatis menggunakan kode python.
    Buatlah visualisasi distribusi menggunakan salah satu jenis (histogram, boxplot, atau statistik deskriptif).
    Buat insight singkat mengenai hasil profiling data.
    Contoh: 
    “10% transaksi bermasalah, mayoritas error di harga dan tanggal”.

    


In [2]:
df = pd.read_csv("orders_large.csv")

In [3]:
len(df)

205

In [4]:
df.isna().sum()

order_id        0
customer_id     0
product_id      8
tanggal         0
jumlah          0
harga           0
email           7
kode_pos       11
status          0
kategori        0
dtype: int64

In [5]:
df['order_id'].duplicated().sum()

np.int64(5)

In [6]:
df['jumlah'] = pd.to_numeric(df['jumlah'], errors='coerce')
df['harga'] = pd.to_numeric(df['harga'], errors='coerce')
df['tanggal'] = pd.to_datetime(df['tanggal'], errors="coerce")

In [7]:
df.isna().sum()

order_id        0
customer_id     0
product_id      8
tanggal        20
jumlah          0
harga           0
email           7
kode_pos       11
status          0
kategori        0
dtype: int64

In [8]:
df['email'].isna().sum()

np.int64(7)

In [9]:
len(df[~df['email'].str.contains('@')])

73

In [10]:
jumlah_invalid = df.loc[df['jumlah'] <= 0, 'jumlah']
harga_invalid = df.loc[(df['harga'] <= 0) | (df['harga'] > 10000000), 'harga' ]
tanggal_invalid = df.loc[df['tanggal'].isna(), 'tanggal']
email_invalid = df[~df['email'].str.contains('@')]


In [11]:
summary = pd.DataFrame({
    'Rule' : ['Jumlah Invalid','Harga Invalid','Tanggal Invalid', 'Email Invalid'],
    'Invalid' : [len(jumlah_invalid),len(harga_invalid),len(tanggal_invalid),len(email_invalid)]
     
})

summary

,Rule,Invalid
0,Jumlah Invalid,13
1,Harga Invalid,12
2,Tanggal Invalid,20
3,Email Invalid,73


In [12]:
print(f'Insight :\nMasalah kualitas data paling besar ada di kolom Email yaitu 73 dari 205 dengan persentase {len(email_invalid)/len(df)*100:.2f} %.\nPola utamanya email tanpa domain, dan email kehilangan tanda "@".\nAngka ini sudah termasuk dengan email kosong')

Insight :
Masalah kualitas data paling besar ada di kolom Email yaitu 73 dari 205 dengan persentase 35.61 %.
Pola utamanya email tanpa domain, dan email kehilangan tanda "@".
Angka ini sudah termasuk dengan email kosong


Question 2. Data Cleaning

Instruksi:

    Hapus record duplicate berdasarkan order_id.
    Normalisasi kategori produk (Elektronik, Electronics, elektronik → elektronik; Fashion, Fesyen → fashion).
    Isi missing kode_pos dengan default 00000.
    Catat jumlah baris sebelum & sesudah cleaning.

Output yang diharapkan:

    Dataset yang telah bersih dapat disimpan ke format csv dengan penamaan “orders_clean.csv”.
    Berikan gambaran dengan menggunakan tabel contoh data  sebelum & sesudah cleaning (5–10 baris).
    Buatkan ringkasan berupa berapa baris data duplicate yang  dihapus, berapa data missing yang diisi.

    

In [13]:
df

,order_id,customer_id,product_id,tanggal,jumlah,harga,email,kode_pos,status,kategori
0,5001,C006,P019,2023-05-04,5,706894,user5001@yahoo.com,16394,paid,Electronics
1,5002,C009,P007,2023-05-03,3,820225,user5002,86978,cancelled,Fesyen
2,5003,C008,P024,2023-05-24,1,131000,user5003,47919,paid,Electronics
3,5004,C022,P024,2023-05-13,-1,508493,user5004@mail.com,93431,pending,Elektronik
4,5005,C043,P020,2023-05-14,-1,250938,user5005,10828,paid,Fesyen
...,...,...,...,...,...,...,...,...,...,...
200,5163,C048,P004,NaT,4,99999999,user5163@gmail.com,69132,paid,elektronik
201,5187,C050,P022,2023-05-28,-1,917806,user5187@gmail.com,23257,paid,Electronics
202,5026,C020,P010,2023-05-07,1,510899,user5026@gmail.com,24321,paid,Electronics
203,5194,C020,P021,2023-05-11,2,742006,user5194,38572,paid,elektronik


In [14]:
df = df.drop_duplicates(subset='order_id')

In [15]:
df['kategori'].unique()

<StringArray>
['Electronics', 'Fesyen', 'Elektronik', 'elektronik', 'Fashion']
Length: 5, dtype: str

In [16]:
df['kategori'] = df['kategori'].str.lower().replace({
    'electronics' : 'elektronik',
    'fesyen' : 'fashion'
})

df['kategori'].unique()

<StringArray>
['elektronik', 'fashion']
Length: 2, dtype: str

In [17]:
df['kode_pos'].isna().sum()

np.int64(11)

In [18]:
df['kode_pos'] = df['kode_pos'].fillna("00000")

In [19]:
df['kode_pos'].isna().sum()

np.int64(0)

In [20]:
df.to_csv("orders_clean.csv", index=False)

In [21]:
print("===== BEFORE ======")

df_before = pd.read_csv("orders_large.csv")
df_before.head(10)

===== BEFORE ======


,order_id,customer_id,product_id,tanggal,jumlah,harga,email,kode_pos,status,kategori
0,5001,C006,P019,2023-05-04,5,706894,user5001@yahoo.com,16394,paid,Electronics
1,5002,C009,P007,2023-05-03,3,820225,user5002,86978,cancelled,Fesyen
2,5003,C008,P024,2023-05-24,1,131000,user5003,47919,paid,Electronics
3,5004,C022,P024,2023-05-13,-1,508493,user5004@mail.com,93431,pending,Elektronik
4,5005,C043,P020,2023-05-14,-1,250938,user5005,10828,paid,Fesyen
5,5006,C048,P029,2023-05-16,2,963485,user5006@gmail.com,79991,paid,Electronics
6,5007,C009,P028,2023-05-06,2,176773,user5007mail.com,58385,pending,Fesyen
7,5008,C049,P005,2023-05-06,2,748857,user5008,74893,pending,elektronik
8,5009,C032,P015,2023-02-31,5,255845,user5009@yahoo.com,85114,pending,Fashion
9,5010,C040,P024,2023-05-10,-1,955406,user5010,83052,pending,Fesyen


In [22]:
print("===== AFTER ======")

df_after = pd.read_csv("orders_clean.csv", dtype={'kode_pos':str})
df_after.head(10)

===== AFTER ======


,order_id,customer_id,product_id,tanggal,jumlah,harga,email,kode_pos,status,kategori
0,5001,C006,P019,2023-05-04,5,706894,user5001@yahoo.com,16394,paid,elektronik
1,5002,C009,P007,2023-05-03,3,820225,user5002,86978,cancelled,fashion
2,5003,C008,P024,2023-05-24,1,131000,user5003,47919,paid,elektronik
3,5004,C022,P024,2023-05-13,-1,508493,user5004@mail.com,93431,pending,elektronik
4,5005,C043,P020,2023-05-14,-1,250938,user5005,10828,paid,fashion
5,5006,C048,P029,2023-05-16,2,963485,user5006@gmail.com,79991,paid,elektronik
6,5007,C009,P028,2023-05-06,2,176773,user5007mail.com,58385,pending,fashion
7,5008,C049,P005,2023-05-06,2,748857,user5008,74893,pending,elektronik
8,5009,C032,P015,NaN,5,255845,user5009@yahoo.com,85114,pending,fashion
9,5010,C040,P024,2023-05-10,-1,955406,user5010,83052,pending,fashion


In [29]:
df_after

,order_id,customer_id,product_id,tanggal,jumlah,harga,email,kode_pos,status,kategori
0,5001,C006,P019,2023-05-04,5,706894,user5001@yahoo.com,16394,paid,elektronik
1,5002,C009,P007,2023-05-03,3,820225,user5002,86978,cancelled,fashion
2,5003,C008,P024,2023-05-24,1,131000,user5003,47919,paid,elektronik
3,5004,C022,P024,2023-05-13,-1,508493,user5004@mail.com,93431,pending,elektronik
4,5005,C043,P020,2023-05-14,-1,250938,user5005,10828,paid,fashion
...,...,...,...,...,...,...,...,...,...,...
195,5196,C006,P025,2023-05-27,5,722763,user5196@gmail.com,87755,pending,elektronik
196,5197,C047,P014,2023-05-30,-1,475376,user5197@gmail.com,66154,pending,fashion
197,5198,C004,P019,2023-05-09,5,127494,user5198@mail.com,31772,pending,elektronik
198,5199,C025,P001,NaN,3,763456,user5199,82055,pending,elektronik


In [24]:
print("===== SUMMARY =====")
print(f"Baris sebelum cleaning : {len(df_before)}")
print(f"Baris sesudah cleaning : {len(df_after)}")
print(f"Jumlah duplikat yang dihapus : {len(df_before) - len(df_after)}")
print(f"Jumlah missing values yang diisi : {df_before['kode_pos'].isna().sum()}")

===== SUMMARY =====
Baris sebelum cleaning : 205
Baris sesudah cleaning : 200
Jumlah duplikat yang dihapus : 5
Jumlah missing values yang diisi : 11


Question 3. Data Validation

Instruksi:
Definisikan aturan validasi schema:

    order_id: unik, tidak null.
    customer_id & product_id: tidak null.
    jumlah: integer, range 1–100.
    harga: integer, 100–10.000.000.
    email: harus valid format.
    tanggal: harus valid (format YYYY-MM-DD, bukan 31/02/2023).
    status: hanya boleh [paid, pending, cancelled].

Jalankan validasi menggunakan SQL (pandasql), Pandera, atau Great Expectations.

Output yang diharapkan:

    Buatlah laporan validasi dengan menunjukkan berapa baris lolos dan berapa gagal.
    Berikan juga contoh baris yang gagal validasi.
    Buatkan penjelasan yang terstruktur dengan menambahkan informasi terkait proses yang dilakukan.


In [33]:
len(df_after.columns)

10

In [9]:
import logging
import pandas as pd
import pandera.pandas as pa
from pandera.errors import SchemaErrors

# --- logging (force=True supaya jalan di Jupyter) ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler("validation.log"), logging.StreamHandler()],
    force=True,
)
log = logging.getLogger("data_quality")

# --- load data ---
FILE = "orders_clean.csv"
df = pd.read_csv(FILE, dtype={"kode_pos": str})
log.info("Mulai validasi %s (%d baris)", FILE, len(df))

# --- fungsi bantu ---
def is_whole(data_kolom):
    return (data_kolom % 1 == 0) | data_kolom.isna()

def valid_date(data_kolom):
    return pd.to_datetime(data_kolom, format="%Y-%m-%d", errors="coerce").notna()

# --- schema ---
schema = pa.DataFrameSchema({
    "order_id":    pa.Column(nullable=False, unique=True),
    "customer_id": pa.Column(nullable=False),
    "product_id":  pa.Column(nullable=False),
    "jumlah": pa.Column(nullable=False, checks=[
        pa.Check(is_whole, error="harus integer"),
        pa.Check.in_range(1, 100)]),
    "harga": pa.Column(nullable=False, checks=[
        pa.Check(is_whole, error="harus integer"),
        pa.Check.in_range(100, 10_000_000)]),
    "email": pa.Column(nullable=False, checks=
        pa.Check.str_matches(r"^[\w\.\-]+@[\w\.\-]+\.\w+$")),
    "tanggal": pa.Column(nullable=False, checks=
        pa.Check(valid_date, error="format YYYY-MM-DD / tanggal tidak valid")),
    "status": pa.Column(nullable=False, checks=
        pa.Check.isin(["paid", "pending", "cancelled"])),
    # aturan tambahan (tidak ada di soal)
    "kode_pos": pa.Column(nullable=False, checks=
        pa.Check.str_matches(r"^\d{5}$", error="kode_pos harus 5 digit angka")),
}, strict=False)

# --- validasi ---
failures = pd.DataFrame()   # default: tidak ada pelanggaran

try:
    schema.validate(df, lazy=True)
except SchemaErrors as err:
    failures = err.failure_cases


2026-10-16 05:23:41,673 | INFO | Mulai validasi orders_clean.csv (200 baris)


In [10]:
print(failures)

    schema_context      column                                      check  \
0           Column  product_id                               not_nullable   
80          Column       email  str_matches('^[\w\.\-]+@[\w\.\-]+\.\w+$')   
93          Column       email  str_matches('^[\w\.\-]+@[\w\.\-]+\.\w+$')   
92          Column       email  str_matches('^[\w\.\-]+@[\w\.\-]+\.\w+$')   
91          Column       email  str_matches('^[\w\.\-]+@[\w\.\-]+\.\w+$')   
..             ...         ...                                        ...   
37          Column       email                               not_nullable   
36          Column       email                               not_nullable   
35          Column       email                               not_nullable   
34          Column       email                               not_nullable   
126         Column    kode_pos               kode_pos harus 5 digit angka   

    check_number      failure_case  index  
0           None               

In [11]:
# laporan + log hasil
failed_idx = sorted(failures["index"].dropna().astype(int).unique()) if len(failures) else []
passed_idx = df.index.difference(failed_idx)

print("===== LAPORAN VALIDASI =====")
print(f"Total baris : {len(df)}")
print(f"Lolos       : {len(passed_idx)}")
print(f"Gagal       : {len(failed_idx)}")

print("\nPelanggaran per kolom & aturan:")
display(failures.groupby(["column", "check"]).size().reset_index(name="jumlah"))

print("\nContoh baris yang LOLOS:")
display(df.loc[passed_idx].head(5))

print("\nContoh baris yang GAGAL:")
display(df.loc[failed_idx].head(10))

# --- catat ke log ---
if len(failures) == 0:
    log.info("Validasi lolos, tidak ada pelanggaran")
else:
    log.warning("%d dari %d baris gagal validasi (%d pelanggaran)",
                len(failed_idx), len(df), len(failures))
    for (col, chk), n in failures.groupby(["column", "check"]).size().items():
        log.warning("kolom=%s | aturan=%s | pelanggaran=%d", col, chk, n)


===== LAPORAN VALIDASI =====
Total baris : 200
Lolos       : 95
Gagal       : 105

Pelanggaran per kolom & aturan:


,column,check,jumlah
0,email,not_nullable,7
1,email,str_matches('^[\w\.\-]+@[\w\.\-]+\.\w+$'),65
2,harga,"in_range(100, 10000000)",11
3,jumlah,"in_range(1, 100)",12
4,kode_pos,kode_pos harus 5 digit angka,5
5,product_id,not_nullable,8
6,tanggal,not_nullable,19



Contoh baris yang LOLOS:


,order_id,customer_id,product_id,tanggal,jumlah,harga,email,kode_pos,status,kategori
0,5001,C006,P019,2023-05-04,5,706894,user5001@yahoo.com,16394,paid,elektronik
5,5006,C048,P029,2023-05-16,2,963485,user5006@gmail.com,79991,paid,elektronik
11,5012,C034,P030,2023-05-05,2,511382,user5012@yahoo.com,51155,cancelled,fashion
15,5016,C024,P007,2023-05-19,1,416114,user5016@gmail.com,85834,paid,elektronik
16,5017,C036,P010,2023-05-01,5,925155,user5017@yahoo.com,64028,cancelled,fashion



Contoh baris yang GAGAL:


,order_id,customer_id,product_id,tanggal,jumlah,harga,email,kode_pos,status,kategori
1,5002,C009,P007,2023-05-03,3,820225,user5002,86978,cancelled,fashion
2,5003,C008,P024,2023-05-24,1,131000,user5003,47919,paid,elektronik
3,5004,C022,P024,2023-05-13,-1,508493,user5004@mail.com,93431,pending,elektronik
4,5005,C043,P020,2023-05-14,-1,250938,user5005,10828,paid,fashion
6,5007,C009,P028,2023-05-06,2,176773,user5007mail.com,58385,pending,fashion
7,5008,C049,P005,2023-05-06,2,748857,user5008,74893,pending,elektronik
8,5009,C032,P015,NaN,5,255845,user5009@yahoo.com,85114,pending,fashion
9,5010,C040,P024,2023-05-10,-1,955406,user5010,83052,pending,fashion
10,5011,C049,P025,2023-05-04,2,0,user5011@yahoo.com,4012a,paid,fashion
12,5013,C031,P018,2023-05-07,5,122606,user5013,49917,cancelled,elektronik


2026-10-16 05:23:49,202 | WARNING | 105 dari 200 baris gagal validasi (127 pelanggaran)
2026-10-16 05:23:49,207 | WARNING | kolom=email | aturan=not_nullable | pelanggaran=7
2026-10-16 05:23:49,210 | WARNING | kolom=email | aturan=str_matches('^[\w\.\-]+@[\w\.\-]+\.\w+$') | pelanggaran=65
2026-10-16 05:23:49,211 | WARNING | kolom=harga | aturan=in_range(100, 10000000) | pelanggaran=11
2026-10-16 05:23:49,214 | WARNING | kolom=jumlah | aturan=in_range(1, 100) | pelanggaran=12
2026-10-16 05:23:49,215 | WARNING | kolom=kode_pos | aturan=kode_pos harus 5 digit angka | pelanggaran=5
2026-10-16 05:23:49,217 | WARNING | kolom=product_id | aturan=not_nullable | pelanggaran=8
2026-10-16 05:23:49,223 | WARNING | kolom=tanggal | aturan=not_nullable | pelanggaran=19



**Proses:**
1. Memuat orders_clean.csv (kode_pos dibaca sebagai string).
2. Mendefinisikan schema pandera sesuai aturan soal.
3. Validasi dengan lazy=True agar semua pelanggaran terkumpul sekaligus.
4. Menghitung baris lolos/gagal dari indeks baris yang melanggar.
5. Mencatat proses dan hasil validasi ke validation.log menggunakan modul logging (INFO untuk proses normal, WARNING untuk pelanggaran).


In [14]:
total_pelanggaran = len(failures)
ringkas = (failures.groupby(["column", "check"]).size()
           .sort_values(ascending=False))

print(f"Lolos: {len(passed_idx)} | Gagal: {len(failed_idx)} | Pelanggaran: {total_pelanggaran}")
print(ringkas)

Lolos: 95 | Gagal: 105 | Pelanggaran: 127
column      check                                    
email       str_matches('^[\w\.\-]+@[\w\.\-]+\.\w+$')    65
tanggal     not_nullable                                 19
jumlah      in_range(1, 100)                             12
harga       in_range(100, 10000000)                      11
product_id  not_nullable                                  8
email       not_nullable                                  7
kode_pos    kode_pos harus 5 digit angka                  5
dtype: int64


Question Bonus 1: Tipe data dan coerce
Pastikan jumlah dan harga bertipe integer, dan ubah otomatis kalau terbaca sebagai teks atau float.


In [15]:
schema = pa.DataFrameSchema({
    "jumlah": pa.Column("Int64", coerce=True, nullable=False),
    "harga":  pa.Column("Int64", coerce=True, nullable=False),
})
#"Int64" (huruf I besar) adalah integer yang boleh berisi kosong. Kalau pakai int biasa, coerce akan error saat ada NaN.


In [16]:
print(df[["jumlah", "harga"]].dtypes)   # sebelum

try:
    df_coerced = schema.validate(df, lazy=True)
    print("Lolos")
except SchemaErrors as err:
    display(err.failure_cases)

print(df_coerced[["jumlah", "harga"]].dtypes)   # sesudah (hanya kalau lolos)

jumlah    int64
harga     int64
dtype: object
Lolos
jumlah    Int64
harga     Int64
dtype: object


Soal 2: Aturan antar kolom
Total transaksi (harga * jumlah) tidak boleh lebih dari 100 juta.


In [17]:
schema = pa.DataFrameSchema(
    {"harga": pa.Column(), "jumlah": pa.Column()},
    checks=pa.Check(lambda row: row["harga"] * row["jumlah"] <= 100_000_000,
                    error="total transaksi > 100 juta"),
)
#Baris dengan NaN dihitung gagal, karena perbandingan dengan NaN bernilai False.


In [19]:
try:
    schema.validate(df, lazy=True)
    print("Lolos, tidak ada total transaksi > 100 juta")
except SchemaErrors as err:
    f = err.failure_cases
    bad = sorted(f["index"].dropna().astype(int).unique())
    print(f"{len(bad)} baris melanggar")
    display(
        df.loc[bad, ["order_id", "harga", "jumlah"]]
          .assign(total=lambda row: row["harga"] * row["jumlah"])
    )

4 baris melanggar


,order_id,harga,jumlah,total
52,5053,99999999,4,399999996
76,5077,99999999,3,299999997
151,5152,99999999,4,399999996
162,5163,99999999,4,399999996


Soal 3: Strict mode
Kolom di luar schema dianggap pelanggaran.


In [25]:
schema_strict = pa.DataFrameSchema({
    "order_id": pa.Column(unique=True),
    "status":   pa.Column(),
}, strict=True)

try:
    schema_strict.validate(df, lazy=True)
    print("Lolos, tidak ada kolom di luar schema")
except SchemaErrors as err:
    display(err.failure_cases[["column", "check", "failure_case"]])  # error: kolom lain tidak ada di schema
#Opsi strict="filter" membuang kolom yang tidak ada di schema, bukan error.


,column,check,failure_case
0,None,column_in_schema,customer_id
1,None,column_in_schema,product_id
2,None,column_in_schema,tanggal
3,None,column_in_schema,jumlah
4,None,column_in_schema,harga
5,None,column_in_schema,email
6,None,column_in_schema,kode_pos
7,None,column_in_schema,kategori


NameError: name 'schema_strict' is not defined

In [22]:
schema_filter = pa.DataFrameSchema({
    "order_id": pa.Column(unique=True),
    "status":   pa.Column(),
}, strict="filter")

df_filtered = schema_filter.validate(df)
print(df_filtered.columns.tolist())   # ['order_id', 'status']

['order_id', 'status']


Soal 4: Format ID
customer_id harus C + 3 angka, product_id harus P + 3 angka, kode_pos harus 5 karakter.

In [ ]:
schema_id = pa.DataFrameSchema({
    "customer_id": pa.Column(checks=pa.Check.str_matches(r"^C\d{3}$")),
    "product_id":  pa.Column(checks=pa.Check.str_matches(r"^P\d{3}$")),
    "kode_pos":    pa.Column(checks=pa.Check.str_length(5, 5)),
})

In [29]:
try:
    schema_id.validate(df, lazy=True)
    print("Lolos, semua format ID sesuai")
except SchemaErrors as err:
    f = err.failure_cases
    display(f.groupby(["column", "check"], dropna=False).size().reset_index(name="jumlah"))
    display(f[["index", "column", "check", "failure_case"]].head(10))

,column,check,jumlah
0,product_id,not_nullable,8


,index,column,check,failure_case
0,19,product_id,not_nullable,NaN
1,27,product_id,not_nullable,NaN
2,71,product_id,not_nullable,NaN
3,127,product_id,not_nullable,NaN
4,131,product_id,not_nullable,NaN
5,137,product_id,not_nullable,NaN
6,139,product_id,not_nullable,NaN
7,146,product_id,not_nullable,NaN


In [31]:
schema_id2 = pa.DataFrameSchema({
    "kode_pos": pa.Column(checks=pa.Check.str_matches(r"^\d{5}$")),
})

try:
    schema_id2.validate(df, lazy=True)
    print("Lolos")
except SchemaErrors as err:
    f = err.failure_cases
    display(f.groupby(["column", "check"], dropna=False).size().reset_index(name="jumlah"))
    display(err.failure_cases[["index", "column", "check", "failure_case"]])

,column,check,jumlah
0,kode_pos,str_matches('^\d{5}$'),5


,index,column,check,failure_case
0,10,kode_pos,str_matches('^\d{5}$'),4012a
1,31,kode_pos,str_matches('^\d{5}$'),4012a
2,81,kode_pos,str_matches('^\d{5}$'),4012a
3,93,kode_pos,str_matches('^\d{5}$'),4012a
4,191,kode_pos,str_matches('^\d{5}$'),4012a
